# 2.2 Skip, state and gate: four questions about the kernel stack

In 2.1 the kernel stack `Kernels(5)` scored highest: five blocks of a kernel, a GELU
and a pool, flattened into a linear head. It bets on locality: every feature is built
from a window of the beat, and the window grows with depth. A single kernel of 35 at
depth 1 (2.1.5's control) did as well, 0.5 points under it. This notebook takes
`Kernels(5)` as its reference all the same, because its questions are about blocks:
stacking more of them, and putting a state or a gate after them. It asks four questions
about that stack, each answered by one new block:

| section | the question about the kernel stack | the block that answers it | goal |
|---|---|---|---|
| 2.2.1 | can it get deeper and still learn? | the skip, `y = T(x) + x` | 2.1 |
| 2.2.2 | can it use what came earlier in the beat, in order, by carrying it along rather than by widening its window? | the state of a recurrent layer | 2.5, 2.7 |
| 2.2.3 | can it learn to let parts of its input through and hold others back? | the gate | 2.6 |
| 2.2.4 | can it do that for whole channels at once? | squeeze-excite | 2.6 |

At the end you can say, for each of the four blocks, what it adds to a kernel stack,
which shape goes in and which comes out, what it costs in weights and seconds, and
whether it paid on these heartbeats. You read that from loss curves and from the
difference in points beside the spread of the runs, as in 2.1. These are the blocks you combine in your own design
(`exercises/designs.py`), so each one gets a reason before it gets a place.

## 2.2.0 Where we stand, the data and the baselines

### 2.2.0.1 Where we stand

| design | macro recall on validation | where |
|---|---|---|
| always N | 0.333 | 2.1.1 |
| lesson 1's head with a hidden layer, on the 187 raw steps | ≈ 0.49 | 2.1.1 |
| 1-nearest-neighbour on the raw beat: the floor | ≈ 0.53 | 2.1.1 |
| one kernel of 35 and a pool, the control of 2.1.5 | ≈ 0.56 | 2.1.5 |
| `Kernels(5)`, the kernel stack of depth 5: **the result to beat** | ≈ 0.56 | 2.1.5 |

A block earns its place when it lifts the macro recall over `Kernels(5)` at a cost in
weights and seconds you are willing to pay.

### 2.2.0.2 The data and the floors

The next cell loads the heartbeats of 2.1 ("The data"). **train** holds the beats the weights
are fitted on; **validation** (`valid`) holds the beats of six other patients, which no
model trains on: every design in this notebook is scored and chosen there. The test
split of 2.1 stays untouched. Then it recomputes the two floors that need no training:
always N and 1-nearest-neighbour. Nothing is trained in this cell.

In [ ]:
import sys

import heartbeats as hb
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from notebooktester import param
from sklearn.neighbors import KNeighborsClassifier
from torch import nn
from vectormesh.components import Conv1D, Gate, Pool1D, Recurrent, Skip

sys.path.insert(0, "../1_representation_and_head")  # lesson 1's block diagrams
import diagrams

logger.remove()
logger.add(sys.stderr, level="WARNING")

EPOCHS = param(12, test=1)  # every comparison except 2.2.1
EPOCHS_DEEP = param(10, test=1)  # 2.2.1, the deep stacks
REPEATS = param(3, test=1)  # runs per arm, each from other random weights
MAX_ROWS = param(None, test=2000)  # train rows; None keeps all of them

train, valid = hb.load(max_rows=MAX_ROWS)
print(f"train {len(train):6d} beats {train.counts()}  the weights are fitted here")
print(
    f"valid {len(valid):6d} beats {valid.counts()}  six other patients, designs are chosen here"
)

always_n = hb.scores(valid.labels.numpy(), np.zeros(len(valid), dtype=int))
knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
knn.fit(train.beats[:, :, 0], train.labels)
nearest = hb.scores(valid.labels.numpy(), knn.predict(valid.beats[:, :, 0]))
for name, found in (("always N", always_n), ("1-nearest-neighbour", nearest)):
    recall = {c: round(r, 2) for c, r in found["recall"].items()}
    print(f"{name:20s} macro recall {found['macro_recall']:.3f}   recall {recall}")

These are the numbers of the table, recomputed: no randomness, so they match 2.1.1.

### 2.2.0.3 The result to beat, trained again here

`Kernels(5)` is the design of 2.1.5. It lives in the helper `heartbeats.py` as
`hb.Kernels(5)`, so this notebook imports it instead of writing it out again. Below is its
diagram, drawn as in lesson 1: a box per step, the shape on every arrow. Every block is a
kernel of 5 taps that turns the channels into 32 (its padding keeps the length), a GELU,
and a max pool of 2 that halves the length. After five blocks 5 positions with 32
channels each are left; `flatten` lays them side by side as 5 × 32 = 160 numbers for the
linear head.

In [ ]:
# kinds of box for this lesson's blocks, next to lesson 1's
diagrams.COLOURS.update(
    kernel="#f6c7a8", pool="#fde9b8", state="#bfe3e0", gate="#f3d9e6", skip="#dccdf0"
)
diagrams.LEGEND.update(
    kernel="kernel block (weights)",
    pool="pool, flatten or mean (no weights)",
    data="input",
    state="recurrent layer",
    gate="gate",
    skip="add (skip)",
)


def kernel_blocks(depth: int, width: int = 32) -> list[diagrams.Block]:
    # the boxes of hb.stack(depth): a kernel, a GELU and a pool of 2 per block
    boxes, length, channels = [diagrams.Block("beat", "(B, 187, 1)", "data")], 187, 1
    for i in range(depth):
        length //= 2
        label = f"block {i + 1}\nkernel 5, {channels} → {width}\nGELU, pool 2"
        boxes.append(diagrams.Block(label, f"(B, {length}, {width})", "kernel"))
        channels = width
    return boxes


def draw(rows: dict[str, list[diagrams.Block]]) -> None:
    diagrams.draw_architectures(rows)
    plt.show()


draw(
    {
        "Kernels(5): the result to beat": kernel_blocks(5)
        + [
            diagrams.Block("flatten", "(B, 160)", "pool"),
            diagrams.Block("Linear\n160 → 3", "(B, 3)", "layer"),
        ]
    }
)
print(f"Kernels(5): {hb.n_params(hb.Kernels(5))} weights")

It is trained again in this notebook for two reasons. Every notebook runs on its own. And
a design is compared against a reference trained under the same conditions, in the same
notebook: the same data, the same number of epochs, the same number of repeats. Here the
comparisons train for `EPOCHS = 12` epochs and `REPEATS = 3` runs. The 12 comes from the
curves of this one reference, so it is a starting point and not a verdict on the other
designs: a design can need many more epochs than a kernel stack, and a design that is
still improving when training stops has not been judged. Every arm below is therefore read
on its own curves first, and a conclusion that holds only within 12 epochs says so. A study
of your own trains as long as its curves ask for.

The runs show how far two trainings of the same design land from each other: the spread
beside every score.

Its number will differ a little from 2.1.5's: other runs from other random weights, and
fewer epochs. That difference is the spread between runs at work.

As in 2.1, `hb.spread` trains `REPEATS` runs, each from other random weights; a run's
score is the mean validation macro recall of its last 3 epochs, since one epoch's number swings
by a few hundredths. `hb.line` prints the mean ± the std of the runs, the training
seconds of one run and, against another result, the difference of the means in points; `hb.curves` draws the train loss and the validation macro recall per epoch,
the line the mean of the runs, the band their range, here with the 1-NN floor dashed.

In [ ]:
reference = hb.spread(lambda: hb.Kernels(5), train, valid, EPOCHS, REPEATS)
print(hb.line("Kernels(5), the result to beat", reference))
print("recall per class:", {c: round(r, 2) for c, r in reference.recall().items()})
hb.curves(
    {"Kernels(5)": reference},
    "The result to beat, trained for EPOCHS epochs: has it finished learning?",
    floor=nearest["macro_recall"],
)

`Kernels(5)` reaches about 0.57 here, with a spread of about ±0.01 between the three runs:
close to 2.1.5's ≈ 0.56, and the gap is the size of that spread. The train loss drops
steeply in two epochs and then keeps falling slowly; the validation macro recall stops
rising after about epoch 7 and then swings by about a hundredth from epoch to epoch. So 12
epochs is enough for this design: more epochs lower the train loss, not the score on
unseen patients. That says nothing about the designs below. Per class it finds nearly every N, about seven in ten V, and almost no S.
Every result below is read against this reference.

## 2.2.1 Depth and the skip

In 2.1.5 depth 5 ended 2.6 points above depth 3, but a single kernel of 35 at depth 1,
the control, did as well as depth 5 (0.5 points apart): the wider window carried the gain,
and depth was not needed for it. Image models stack fifty blocks and more. Does depth
help when it is pushed much further, and what happens when you stack many blocks?

To test depth alone, the blocks must keep the shape. The pools of `Kernels` halve the
length after every block, so after five blocks almost nothing is left to stack on.
`Deep(n)` therefore shortens the beat once, in a stem, and then stacks `n` blocks without
pools:

- **stem**: a kernel 1 → 32 with a GELU and a pool, then a second pool:
  `(B, 187, 1) -> (B, 93, 32) -> (B, 46, 32)`. The second pool has no kernel before it: it
  only halves the length once more, so that every block below runs on 46 positions, the
  length 2.2.2 also uses, at half the cost of 93; a kernel there would be one more block;
- **n blocks**, each a kernel 32 → 32 with padding and a GELU, no pool:
  `(B, 46, 32) -> (B, 46, 32)`, so any number of them can follow each other;
- **head**: flatten 46 × 32 = 1472 numbers, then `Linear(1472, 3)`.

The argument `wrap` is empty for now; later in this section it puts something around
every block.

In [ ]:
class Deep(nn.Module):
    # A stem, n blocks that keep the shape (B, 46, 32), and a flatten head.
    # wrap(block, width), when given, is put around every block.

    def __init__(self, n: int, wrap=None, width: int = 32):
        super().__init__()
        # (B, 187, 1) -> (B, 93, 32) -> (B, 46, 32)
        self.stem = nn.Sequential(*hb.stack(1, width), Pool1D(2))
        blocks = []
        for _ in range(n):
            block = nn.Sequential(Conv1D(width, width, 5, padding=2), nn.GELU())
            blocks.append(wrap(block, width) if wrap else block)
        self.blocks = nn.Sequential(*blocks)  # (B, 46, 32) -> (B, 46, 32)
        self.head = nn.Linear(46 * width, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)).flatten(1))


stem = [
    diagrams.Block("beat", "(B, 187, 1)", "data"),
    diagrams.Block(
        "stem\nkernel 5, 1 → 32, GELU\npool 2, pool 2", "(B, 46, 32)", "kernel"
    ),
]
head = [
    diagrams.Block("flatten", "(B, 1472)", "pool"),
    diagrams.Block("Linear\n1472 → 3", "(B, 3)", "layer"),
]
block = "kernel 5, 32 → 32\nGELU"
draw(
    {
        "Deep(2)": stem
        + [
            diagrams.Block(f"block {i}\n{block}", "(B, 46, 32)", "kernel")
            for i in (1, 2)
        ]
        + head,
        "Deep(16)": stem
        + [
            diagrams.Block(f"block 1\n{block}", "(B, 46, 32)", "kernel"),
            diagrams.Block("blocks 2 … 15", "(B, 46, 32)", "kernel"),
            diagrams.Block(f"block 16\n{block}", "(B, 46, 32)", "kernel"),
        ]
        + head,
    }
)
for n in (2, 16):
    print(f"Deep({n:2d}): {hb.n_params(Deep(n)):6d} weights")

### 2.2.1.1 What goes wrong in a deep stack

Stack 16 of these blocks, `Deep(16)`, and the model may learn nothing at all. You can see
why before training, in what happens to a beat on its way through the blocks of a freshly
made model, with its random starting weights. Two things are worth measuring.

**1. Do the beats still differ from each other?** The head can tell an N from a V only if
the numbers it receives differ from one beat to the next. Push 256 beats through the
blocks. After a block, look at one position and one channel: each of the 256 beats gives
one number there, and their std says how much the beats differ at that spot. Three beats
that give 0.9, 0.1 and 0.5 differ (std 0.4); three that give 0.501, 0.499 and 0.500 are
the same beat for all practical purposes (std 0.001). Take the mean of these stds over all
46 positions and 32 channels: that is the **spread across the batch**, one number per
block. A spread of 0 means every beat comes out as the same numbers. If that happens
before the head, the head gets the same 1472 numbers for every beat and can learn only how
common each class is.

**2. Does the loss reach the early blocks?** A weight changes in proportion to the
gradient of the loss on it (lesson 1). Compute the loss on those 256 beats, run one
backward pass, and take the length of the gradient on all the weights of a block: how hard
the loss pulls on that block. Next to the head it pulls hard. If it pulls with a length of
almost 0 on block 1, block 1 stays as random as it started, whatever the data says.

A **skip** is the standard answer to both. It comes first, then the two measures on a
plain stack and a stack with skips, side by side.

### 2.2.1.2 The skip

<svg width="620" height="180" viewBox="0 0 620 180" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="620" height="180" fill="white"/>
<defs><marker id="a1" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="114" fill="#222">x</text>
<line x1="25" y1="110" x2="198" y2="110" stroke="#444" stroke-width="1.5" marker-end="url(#a1)"/>
<text x="70" y="102" font-family="monospace" font-size="11" fill="#222">(B, 46, 32)</text>
<rect x="200" y="88" width="130" height="44" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="265" y="108" text-anchor="middle" fill="#222">block T</text>
<text x="265" y="124" text-anchor="middle" font-size="11" fill="#222">kernel 5, GELU</text>
<line x1="330" y1="110" x2="453" y2="110" stroke="#444" stroke-width="1.5" marker-end="url(#a1)"/>
<text x="390" y="102" text-anchor="middle" fill="#222">T(x)</text>
<text x="390" y="128" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 46, 32)</text>
<circle cx="470" cy="110" r="16" fill="#dccdf0" stroke="#555"/>
<text x="470" y="115" text-anchor="middle" font-size="18" fill="#222">+</text>
<line x1="486" y1="110" x2="568" y2="110" stroke="#444" stroke-width="1.5" marker-end="url(#a1)"/>
<text x="576" y="114" fill="#222">y</text>
<text x="527" y="102" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 46, 32)</text>
<path d="M 110 110 L 110 40 L 470 40 L 470 92" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#a1)"/>
<text x="290" y="32" text-anchor="middle" fill="#222">the skip: x itself, (B, 46, 32), no weights</text>
<text x="310" y="168" text-anchor="middle" fill="#222">y = T(x) + x</text>
</svg>

A **skip** is a route around a block whose output is added back: `y = T(x) + x`, where
`T` is what the block does (slide 11). Three things follow from that one addition:

- **The shapes must match.** `T(x)` and `x` are added element by element, so a block with a
  skip must give `(B, 46, 32)` for `(B, 46, 32)`. That is why the blocks of `Deep` keep the
  shape.
- **The block only learns a correction.** When `T(x)` is close to zero, `y` is close to
  `x`: the input passes unchanged. A block has to learn only what to add.
- **The `+ x` route carries the signal and the gradient past every block.** It has no
  weights, so the differences between beats reach the head through it however much the
  blocks shrink their own output. On the way back the same holds: the derivative of `x`
  with respect to `x` is 1, so the gradient reaches block 1 without being multiplied by
  the weights of the 15 blocks above it.

He and colleagues introduced it in 2015 (ResNet) to train image models of 152 layers;
since then nearly every architecture has one: WaveNet for audio, every block of a
transformer, N-BEATS for series (slide 11). The `skip` switch of lesson 1's search
(notebook 1.5) was a variant: its route went through a projection, a `Linear` with weights
of its own.

The skip is a few lines of code, `PlainSkip` below; it puts `x + T(x)` around every block.

In [ ]:
class PlainSkip(nn.Module):
    # y = x + T(x): slide 11's skip, nothing on the route around the block

    def __init__(self, block: nn.Module):
        super().__init__()
        self.block = block

    def forward(self, x):
        return x + self.block(x)  # (B, 46, 32) + (B, 46, 32)


def plain_skip(block: nn.Module, width: int) -> nn.Module:
    return PlainSkip(block)  # T(x) + x

In [ ]:
STYLES = ["-", "--", ":", "-.", (0, (5, 1, 1, 1))]
ARMS: dict[
    str, int
] = {  # arm name -> its number, so an arm keeps its colour in every figure
    "16 plain blocks": 0,
    "16 blocks, LayerNorm only": 1,
    "16 blocks, PlainSkip": 2,
    "16 blocks, vectormesh Skip": 3,
}


def arm_style(name: str) -> dict:
    i = ARMS.setdefault(name, len(ARMS))
    return dict(color=f"C{i % 10}", ls=STYLES[i % len(STYLES)])


def look_inside(model: Deep, beats: torch.Tensor, labels: torch.Tensor):
    # The spread across the batch and the mean size |h| after the stem and after every
    # block, and the size of the gradient on every block's weights, for one batch.
    model = model.cpu()
    model.zero_grad()
    h = model.stem(beats)
    spread = [h.std(dim=0).mean().item()]
    size = [h.abs().mean().item()]
    for block in model.blocks:
        h = block(h)
        spread.append(h.std(dim=0).mean().item())
        size.append(h.abs().mean().item())
    loss = nn.functional.cross_entropy(model.head(h.flatten(1)), labels)
    loss.backward()
    grad = [
        torch.cat([p.grad.flatten() for p in block.parameters()]).norm().item()
        for block in model.blocks
    ]
    return spread, size, grad


def plot_inside(models: dict[str, Deep], title: str, sizes: bool = False) -> None:
    # sizes=True adds a third panel: the mean size |h| of the numbers after each block
    beats, labels = train.beats[:256], train.labels[:256]
    fig, axes = plt.subplots(1, 3 if sizes else 2, figsize=(17 if sizes else 12, 4))
    left, right = axes[0], axes[-1]
    for name, model in models.items():
        spread, size, grad = look_inside(model, beats, labels)
        style = dict(marker="o", label=name, **arm_style(name))
        left.plot(range(len(spread)), spread, **style)
        right.plot(range(1, len(grad) + 1), grad, **style)
        if sizes:
            axes[1].plot(range(len(size)), size, **style)
    if sizes:
        axes[1].set_title(
            "size of the numbers after each block\n(mean of |h| over 256 beats, positions and channels)",
            fontsize=10,
        )
        axes[1].set_xlabel("block (0 = after the stem)")
        axes[1].set_ylabel("mean |h| over 256 beats (log)")
    left.set_title(
        "spread across the batch: how much the 256 beats differ\n(std over the beats, mean over positions and channels; 0 = all alike)",
        fontsize=10,
    )
    left.set_xlabel("block (0 = after the stem)")
    left.set_ylabel("spread = std over the beats (log)")
    right.set_title(
        "gradient on each block: how hard the loss pulls on its weights\n(length of the gradient; 0 = the block cannot learn)",
        fontsize=10,
    )
    right.set_xlabel("block (1 = nearest the stem)")
    right.set_ylabel("gradient length (log)")
    for ax in axes:
        ax.set_yscale("log")
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


torch.manual_seed(0)
plot_inside(
    {"16 plain blocks": Deep(16), "16 blocks, PlainSkip": Deep(16, plain_skip)},
    "Before any training: 16 plain blocks and 16 blocks with a skip",
)

Read the plain stack first (blue). After the stem the beats differ by about 0.03. After
block 8 that is about 2 × 10⁻⁶ and after block 16 about 7 × 10⁻¹⁰: every beat reaches the
head as the same 1472 numbers, and a head that sees the same input for every beat can learn
only one thing, the class shares. The gradient tells the same story from the other side:
next to the head (block 16) the loss pulls on the weights with a length of about 0.2, on
block 1 with about 6 × 10⁻⁹. Block 1 hardly changes, whatever the data says. (Adam, the
optimizer of `hb.train`, divides a step by the size of the weight's own recent gradients,
so a small gradient is less fatal than under plain gradient descent, but at this size it
does not rescue block 1.)

Why every block shrinks the differences: its kernel weights start small and random, and a
GELU passes only about half of a small value (its slope at 0 is 0.5, notebook 1.2). A block
keeps a part of the differences between beats, and 16 parts of parts leave almost nothing.
On the way back the gradient is multiplied by the same small weights and slopes, block
after block: the vanishing gradient lesson 1 described, here measured.

Now the stack with a skip (orange). The spread does not fall: about 0.03 after the stem,
about 0.07 after block 16. The gradient is about 1 on block 1 and about 1 on block 16. That
is what the `+ x` route does: the differences between beats and the gradient go past the
blocks through it. This is a picture of the start only, before any weight has changed;
whether the stack learns is the experiment below.

In [ ]:
shares = np.bincount(train.labels.numpy(), minlength=3) / len(train)
class_share_loss = float(-(shares * np.log(shares)).sum())
print(
    "class shares in train:",
    {c: round(float(p), 3) for c, p in zip(hb.CLASSES, shares)},
)
print(f"loss of a model that only knows the class shares: {class_share_loss:.3f}")

A train loss stuck at that level is a model that knows the class shares and nothing else.

**Before you read on**, write down what you expect the train loss of `Deep(16)` to do in
its first epochs, given the picture above, with and without a skip, and what you expect of `Deep(2)`.

### 2.2.1.3 Is a norm enough, or does it take the add?

The skip above is a few lines of code, `PlainSkip` below. vectormesh's `Skip` does one
thing more: it applies a **LayerNorm** to the input first, `y = T(LN(x)) + LN(x)`. A
LayerNorm takes the 32 channels at one position of one beat and rescales them to mean 0
and spread 1, then multiplies by a learned scale and adds a learned shift (which start at
1 and 0). Whatever the blocks before it did to the size of the numbers, the next block
gets numbers of a normal size. So the route of `Skip` around each block skips the kernel
but still passes through a LayerNorm, with weights of its own; the route of `PlainSkip`
has none. To separate the add from the norm, the experiment has both, and a third arm
with the LayerNorm before every block and no add: `y = T(LN(x))`.

In [ ]:
def with_skip(block: nn.Module, width: int) -> nn.Module:
    return Skip(block, in_size=width)  # vectormesh: T(LN(x)) + LN(x)


def norm_only(block: nn.Module, width: int) -> nn.Module:
    return nn.Sequential(nn.LayerNorm(width), block)  # T(LN(x)), no add


torch.manual_seed(0)
plot_inside(
    {
        "16 plain blocks": Deep(16),
        "16 blocks, LayerNorm only": Deep(16, norm_only),
        "16 blocks, PlainSkip": Deep(16, plain_skip),
        "16 blocks, vectormesh Skip": Deep(16, with_skip),
    },
    "Before any training: the four 16-block designs",
)

All three changes fix the start: with the LayerNorm alone, with `PlainSkip` and with
vectormesh's `Skip`, the spread stays between about 0.03 and 0.3 through all 16 blocks,
and every block gets a gradient of about 1 to 10. The LayerNorm alone does it by rescaling
the numbers before every block; the skips by adding the input back. Judged by this
picture, all three should train. The picture shows only the first step, though, before any
weight has changed; whether they train is the experiment.

### 2.2.1.4 Does a deep stack train, and does it pay?

**Question**: does a stack of 16 blocks learn, with and without a skip, and does it beat 2
blocks?

**Arms**: `Deep(2)` plain; `Deep(16)` plain; `Deep(16)` with a LayerNorm before every
block and no add; `Deep(16)` with `PlainSkip` around every block; `Deep(16)` with
vectormesh's `Skip` around every block.

**Setup**: every arm trains for `EPOCHS_DEEP = 10` epochs, `REPEATS` runs each. A stack of
16 blocks has six times the weights of `Deep(2)` and may need epochs before it starts to
learn at all. Ten is enough here to see whether each arm starts and whether its loss
flattens; the curves are the check. Each run is drawn as its own line, so a run that
starts late shows. The dashed line on the left is the loss of a model that only knows the
class shares; on the right, the macro recall of always N. After training, the first run of
every 16-block arm is looked inside again, as before training.

**Before you run the next cell**, write down for every arm whether it will start learning,
and whether 16 blocks with a skip will beat 2 plain blocks.

In [ ]:
def run_curves(results: dict[str, hb.Spread], title: str) -> None:
    # Train loss and validation macro recall per epoch, one line per run, with the levels
    # of a model that only knows the class shares dashed.
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
    for name, result in results.items():
        for j, run in enumerate(result.runs):
            epochs = np.arange(1, len(run.train_loss) + 1)
            label = name if j == 0 else None
            style = dict(lw=1.5, label=label, **arm_style(name))
            left.plot(epochs, run.train_loss, **style)
            right.plot(epochs, run.val_recall, **style)
    left.axhline(class_share_loss, color="0.4", ls="--", lw=1)
    left.text(
        1,
        class_share_loss * 1.12,
        "knows only the class shares",
        color="0.3",
        fontsize=9,
    )
    right.axhline(1 / 3, color="0.4", ls="--", lw=1)
    right.text(1, 1 / 3 + 0.005, "always N", color="0.3", fontsize=9)
    left.set_yscale("log")
    left.set_title("train loss: when does it start, has it stopped falling?")
    left.set_xlabel("epoch")
    left.set_ylabel("cross-entropy on train (log)")
    right.set_title("macro recall on unseen patients")
    right.set_xlabel("epoch")
    right.set_ylabel("macro recall (validation)")
    left.legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


arms = {
    "2 plain blocks": lambda: Deep(2),
    "16 plain blocks": lambda: Deep(16),
    "16 blocks, LayerNorm only": lambda: Deep(16, norm_only),
    "16 blocks, PlainSkip": lambda: Deep(16, plain_skip),
    "16 blocks, vectormesh Skip": lambda: Deep(16, with_skip),
}
trained = {}  # the models each spread trains, kept to look inside afterwards


def keeping(name, make):
    def make_and_keep():
        trained.setdefault(name, []).append(make())
        return trained[name][-1]

    return make_and_keep


deep = {
    name: hb.spread(keeping(name, make), train, valid, EPOCHS_DEEP, REPEATS)
    for name, make in arms.items()
}
for name, make in arms.items():
    versus = None if name == "2 plain blocks" else deep["2 plain blocks"]
    print(f"{hb.line(name, deep[name], versus)}   {hb.n_params(make()):6d} weights")
print("pt: the difference with 2 plain blocks, in points of macro recall")
run_curves(deep, "Deep(n) for EPOCHS_DEEP epochs, every run its own line")

Compare with what you wrote down.

- **16 plain blocks** (blue) never starts. In all three runs the train loss lies on the
  dashed line of a model that knows only the class shares, about 0.34, for all 10 epochs,
  and the macro recall is 0.333: it says N for every beat. The picture of the start came
  true in training. A stack on such a plateau can leave it by chance after many more
  epochs; within this budget none of the three runs did.
- **16 blocks with the LayerNorm only** (orange) stays on the same plateau in all three
  runs, although its start looked as healthy as the skips'.
- **16 blocks with `PlainSkip`** (green), slide 11's `x + T(x)` with nothing on the
  route, learns from the first epoch, and so does **vectormesh's `Skip`** (red). Their train
  losses end at about 0.02, against about 0.04 for 2 plain blocks, and are still falling
  at epoch 10, while their validation macro recall is flat from about epoch 6. So the add is what makes
  this stack train; the norm inside `Skip` is not needed for it.
- **Does depth pay?** On unseen patients the 16-block stacks with a skip end level with 2
  plain blocks: `PlainSkip` a little under, `Skip` a little over, each closer to it than
  the spread between their runs. The 16 blocks fit the
  training beats better, and that does not carry over to the six other patients. They cost
  six times the weights (about 88,000 against 14,915) and three to four times the seconds
  per run.

Depth bought nothing on these beats, as depth was not needed in 2.1.5 once a kernel was
wide enough. What the skip buys is that depth becomes possible at all; where depth pays, in
large image models, the skip is what makes those models trainable.

### 2.2.1.5 What separates the norm from the skip

Before training, the LayerNorm-only stack looked as healthy as the two skips. The cell
below looks inside the first trained run of every 16-block arm, after its `EPOCHS_DEEP`
epochs, in the same way as before training, with one panel more in the middle: the mean
size `|h|` of the numbers after each block. A spread that falls can mean two things: the
numbers all become small, or they stay large and become the same for every beat. The
middle panel tells the two apart.

In [ ]:
plot_inside(
    {name: models[0] for name, models in trained.items() if name.startswith("16")},
    "The 16-block designs after training (first run of each)",
    sizes=True,
)

Now the LayerNorm-only stack and the skips come apart. In the first run of the
LayerNorm-only stack (the curves above show all three runs on the plateau) the spread
across the batch falls again, block by block, from about 0.1 after block 1 to about
2 × 10⁻⁵ after block 16, and the gradient on block 1 is about 10⁻⁴ against about 1 on block
16. The middle panel shows that the numbers did not become small: their mean size stays
between about 0.1 and 2 through all 16 blocks. They became the same for every beat. The
stack collapsed during training, in the forward pass as the plain stack was collapsed from
the start. In both skip stacks the spread grows from about 0.03 after the stem to about
0.5 by block 16, and the gradient is about the same on every block (for `Skip` a little
higher on block 1). Its level
differs between designs (about 2 × 10⁻³ on every block of `PlainSkip`, about 0.07 for
`Skip`, at about the same train loss), so compare the shape of each line, not its height.

A LayerNorm works on one beat at a time: it gives the 32 numbers of a position mean 0 and
spread 1, but if every beat comes out of a block with the same pattern, it rescales the
same pattern for every beat, and the differences between beats stay lost. A skip adds the
block's input back, so the differences that reached a block also leave it, whatever the
block does. That is what this picture shows. What drove the LayerNorm-only stack into its
collapse in the first steps of training, it does not show.

## 2.2.2 The state: relating parts of the beat in order

**The axis.** The axis this section is about is the time axis `S` of the beat. After
the first two blocks of the kernel stack, `hb.stack(2)`, a beat is `(B, 46, 32)`: 46
positions in time, each standing for 4 steps of the raw beat (32 ms), and at every
position 32 channels, the answers of the 32 kernels of block 2.

**What a kernel sees.** A position after those two blocks depends on a window of 16 raw
steps (2.1.4), about 0.13 s: its receptive field. Of everything outside that window it
knows nothing.

**Context.** On slide 9 the same diver means one thing on a diving board and another in a
classroom: the value is the same, what came before it differs. A beat has the same
problem. A steep rise right after the R-peak is part of the beat itself; the same steep
rise 0.6 s later is the start of the next beat. What came earlier in the sequence is the
**context** of a value.

**Two ways to reach it.** A kernel stack reaches context by widening its window: every
block adds to the receptive field, and a larger kernel or a dilation adds more. That works at
any distance the window covers, runs all positions in parallel, and a gradient from a far
position travels only through the depth of the stack. A recurrent layer reaches context by
carrying it: one vector that every step updates, so nothing needs a window, but the gradient
from position 1 to position 46 travels through 45 updates, and that long path is where
gradients vanish. Neither is the only way to relate parts that lie far apart; the
difference is how, and what each costs.

**The state.** A **recurrent layer** walks the 46 positions in order, from the first to
the last, and keeps one vector of `H = 64` numbers: the **state** `h`. At every step it
computes a new state from the new input and the old state, with the same weights at
every step: `h_t = f(x_t, h_{t-1})`. The state starts as zeros, `h_0`. After step `t` it
has seen `x_1 … x_t`, in that order; after step 46 it has seen the whole beat.

<svg width="720" height="210" viewBox="0 0 720 210" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="210" fill="white"/>
<defs><marker id="a2" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="20" y="74" fill="#222">h₀ = 0</text>
<line x1="70" y1="70" x2="108" y2="70" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<rect x="110" y="50" width="60" height="40" rx="8" fill="#bfe3e0" stroke="#555"/><text x="140" y="75" text-anchor="middle" fill="#222">f</text>
<line x1="170" y1="70" x2="248" y2="70" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<text x="209" y="62" text-anchor="middle" fill="#222">h₁</text>
<rect x="250" y="50" width="60" height="40" rx="8" fill="#bfe3e0" stroke="#555"/><text x="280" y="75" text-anchor="middle" fill="#222">f</text>
<line x1="310" y1="70" x2="388" y2="70" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<text x="349" y="62" text-anchor="middle" fill="#222">h₂</text>
<rect x="390" y="50" width="60" height="40" rx="8" fill="#bfe3e0" stroke="#555"/><text x="420" y="75" text-anchor="middle" fill="#222">f</text>
<line x1="450" y1="70" x2="488" y2="70" stroke="#444" stroke-width="1.5"/>
<text x="510" y="75" text-anchor="middle" fill="#222">…</text>
<line x1="530" y1="70" x2="568" y2="70" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<rect x="570" y="50" width="60" height="40" rx="8" fill="#bfe3e0" stroke="#555"/><text x="600" y="75" text-anchor="middle" fill="#222">f</text>
<line x1="630" y1="70" x2="678" y2="70" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<text x="684" y="74" fill="#222">h₄₆</text>
<text x="209" y="88" text-anchor="middle" font-family="monospace" font-size="10" fill="#222">(B, 64)</text>
<text x="652" y="88" text-anchor="middle" font-family="monospace" font-size="10" fill="#222">(B, 64)</text>
<line x1="140" y1="150" x2="140" y2="92" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<line x1="280" y1="150" x2="280" y2="92" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<line x1="420" y1="150" x2="420" y2="92" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<line x1="600" y1="150" x2="600" y2="92" stroke="#444" stroke-width="1.5" marker-end="url(#a2)"/>
<text x="140" y="168" text-anchor="middle" fill="#222">x₁</text>
<text x="280" y="168" text-anchor="middle" fill="#222">x₂</text>
<text x="420" y="168" text-anchor="middle" fill="#222">x₃</text>
<text x="600" y="168" text-anchor="middle" fill="#222">x₄₆</text>
<text x="140" y="184" text-anchor="middle" font-family="monospace" font-size="10" fill="#222">(B, 32)</text>
<text x="280" y="184" text-anchor="middle" font-family="monospace" font-size="10" fill="#222">(B, 32)</text>
<text x="420" y="184" text-anchor="middle" font-family="monospace" font-size="10" fill="#222">(B, 32)</text>
<text x="600" y="184" text-anchor="middle" font-family="monospace" font-size="10" fill="#222">(B, 32)</text>
<text x="360" y="25" text-anchor="middle" fill="#222">one recurrent layer, unrolled: the same f, with the same weights, at every one of the 46 steps</text>
<text x="360" y="204" text-anchor="middle" fill="#222">x_t: the 32 channels at position t   ·   h_t: the state after step t, 64 numbers</text>
</svg>

The picture is one layer drawn out over time: there is one `f`, with one set of weights,
applied 46 times. The simplest `f` is one layer of lesson 1 with two inputs:

`h_t = tanh(W x_t + U h_{t-1} + b)`, with `W` of 64 × 32, `U` of 64 × 64 and `b` of 64.

`W` turns the 32 channels of the new position into 64 numbers, `U` turns the old state
into 64 numbers, and the tanh squeezes their sum into the new state. That is where 32
becomes 64. A **GRU** (gated recurrent unit), the recipe used below, is this with gates
added: it has three such sets of weights, two for its gates and one for the new candidate
state, each with two biases in PyTorch, so `3 × (64 × 32 + 64 × 64 + 2 × 64) = 18,816`
weights. 2.2.3 explains what a gate is; you do not need the GRU's insides for this section.
vectormesh's `Recurrent(in_size, hidden_size, kind="gru")` runs PyTorch's `nn.GRU` over
the sequence.

### 2.2.2.1 The shapes, one arrow at a time

The cell below pushes 8 random "beats after two kernel blocks" through a GRU and a linear
head, and prints the shape after every step.

In [ ]:
print(f"receptive field of hb.stack(2): {hb.receptive_field(hb.stack(2))} raw steps")
x = torch.randn(8, 46, 32)  # 8 beats, 46 positions, 32 channels
gru = Recurrent(in_size=32, hidden_size=64, kind="gru")
states = gru(x)  # the state after every step
last = states[:, -1]  # the state after the last step
logits = nn.Linear(64, len(hb.CLASSES))(last)
print(f"x        {tuple(x.shape)}")
print(f"states   {tuple(states.shape)}")
print(f"last     {tuple(last.shape)}")
print(f"logits   {tuple(logits.shape)}")
print(f"weights of the GRU: {hb.n_params(gru)}")

Per arrow:

- `(B, 46, 32) -> GRU -> (B, 46, 64)`: the batch and the 46 positions stay. The GRU
  hands back its state after every step: 46 states per beat, each of 64 numbers. The 32
  channels of a position go in at that step; the GRU's weights turn them, together with
  the old state, into 64 numbers. 64 is `H`, the hidden size, which you choose, as you
  chose the width of a hidden layer in lesson 1.
- `[:, -1] -> (B, 64)`: keep, for every beat, the state after the last step: index `-1`
  on axis 1, the time axis. The axis is gone. It is the state to keep, because it is the
  only one that has seen all 46 positions.
- `Linear(64, 3) -> (B, 3)`: the head, as in lesson 1.

Why not flatten `(B, 46, 64)` into `(B, 2944)`, as the kernel stack flattens its output?
A flatten gives the head every position separately, with its own weights for each one: the
head would then relate the positions itself, and the state would have no job. Taking one
state asks the GRU to carry what matters to the end.

### 2.2.2.2 What a state can, and what it costs

All three designs of this notebook see **order**, each in its own way:

- **A kernel** sees order only within its field: inside its window of 16 raw steps every
  step has its own weight, beyond the window it sees nothing.
- **The flatten head** sees every position, and has a fixed weight for each, so it knows
  *where* in the beat something happened. But it combines the positions linearly and all
  at once: it multiplies each number by its weight and adds them up. What position 30
  adds to the score is the same whatever happened at position 5.
- **A state** reads the positions one after another, so an early event can change how a
  later one is read: the state after step 5 is part of the input of step 30. That works
  nonlinearly (the tanh and the gates at every step) and **at any distance**: the state
  after step 46 can still depend on step 1. Swap two inputs and the state ends up
  elsewhere (slide 10).

It costs two things. Step 10 waits for step 9: the 46 steps cannot be computed side by
side, as the windows of a kernel are. And **the past fades**: every step writes over part
of the state, so what came from `x_1` thins out by step 46 (slide 10). The first cost
shows in the seconds of the experiment below: its arms have the same kernels, on the same
device and the same 46 positions, and only some have a GRU.

### 2.2.2.3 The zero tail

Every beat runs to 1.2 times the time between two beats after its R-peak, and is then
padded with zeros up to 187 steps (2.1, "The data"). The cell below counts, per beat of
validation, how many of the 46 positions after the two pools still hold the beat.

In [ ]:
beats = valid.beats[:, :, 0].numpy()
real = np.array([np.flatnonzero(b).max() + 1 for b in beats])  # steps before the zeros
held = real // 4  # positions after two pools of 2
q10, q50, q90 = np.percentile(held, [10, 50, 90])
print(
    f"positions of 46 that hold the beat: median {q50:.0f}, 10% of beats {q10:.0f} or fewer, 10% {q90:.0f} or more"
)

In the median beat only 26 of the 46 positions hold the beat; the other 20 are padding.
For one beat in ten, 19 or fewer positions hold the beat, so the GRU walks 27 or more
steps of padding after the beat has ended; for another one in ten, 42 or more hold it.
After the kernels the padding is not zero but the same input at every step (what the
kernels make of zeros), and it carries nothing about the beat. The last state is read
after that tail, whose length depends on the heart rate, so the GRU has to carry what it
saw across it. That is the reason for a second way to read the GRU below: the mean of its
states at the positions that hold the beat, so that the tail does not count.

### 2.2.2.4 Does a state add anything on one beat?

A state bets on **order**: what came before changes what this means (slide 18). The
question here is narrower: **does a state add anything over a head that already sees
every position of the beat?** The arms keep the kernels and change only what comes after
them. All start with the first two blocks of the kernel stack, `hb.stack(2)`,
`(B, 187, 1) -> (B, 46, 32)`:

- **2 blocks + flatten**: the same kernels, no state, the flatten head of `Kernels`. The
  control: what the GRU adds is read against this one.
- **2 blocks + GRU, last state**: the design of the shapes above.
- **2 blocks + GRU, mean over the beat**: the same GRU, its states averaged over the
  positions that hold the beat, `(B, 46, 64) -> (B, 64)`. The states of the tail are left
  out, so this arm does not depend on the tail being crossed.
- **`Kernels(5)`**, the result to beat, trained above under the same conditions.

What this experiment can test, given the three ways of seeing order above: every arm knows
where in the beat something happened, so none is blind to order. What only the state adds
is that an early part of the beat can change how a later part is read. If a GRU arm beats
the control, that kind of relation carried something the linear head could not reach. If
it does not, there are two readings that this experiment cannot tell apart: such relations
carry nothing on one beat, or the GRU did not find them within 12 epochs. And it cannot
test whether the order of *beats* matters: a row holds one beat.

In [ ]:
class TwoBlocksThen(nn.Module):
    # hb.stack(2), (B, 187, 1) -> (B, 46, 32), then one of three ways to (B, 3)

    def __init__(self, read: str, width: int = 32, hidden: int = 64):
        super().__init__()
        self.read = read
        self.features = nn.Sequential(*hb.stack(2, width))
        if read == "flatten":
            self.head = nn.Linear(46 * width, len(hb.CLASSES))
        else:  # "GRU, last state" or "GRU, mean over the beat"
            self.state = Recurrent(width, hidden, kind="gru")
            self.head = nn.Linear(hidden, len(hb.CLASSES))

    def forward(self, x):
        h = self.features(x)  # (B, 46, 32)
        if self.read == "flatten":
            return self.head(h.flatten(1))  # (B, 1472)
        states = self.state(h)  # (B, 46, 64)
        if self.read == "GRU, last state":
            return self.head(states[:, -1])  # (B, 64)
        # (B, 46): True where the 4 raw steps under a position are not all padding
        holds = x[:, : 46 * 4, 0].reshape(len(x), 46, 4).abs().sum(-1) > 0
        weight = holds.float().unsqueeze(-1)  # (B, 46, 1)
        mean = (states * weight).sum(dim=1) / weight.sum(dim=1).clamp(min=1)
        return self.head(mean)  # (B, 64)


two = kernel_blocks(2)
gru_box = diagrams.Block("GRU\n32 → 64", "(B, 46, 64)", "state")
draw(
    {
        "2 blocks + flatten": two
        + [
            diagrams.Block("flatten", "(B, 1472)", "pool"),
            diagrams.Block("Linear\n1472 → 3", "(B, 3)", "layer"),
        ],
        "2 blocks + GRU, last state": two
        + [
            gru_box,
            diagrams.Block("[:, -1]\nlast state", "(B, 64)", "pool"),
            diagrams.Block("Linear\n64 → 3", "(B, 3)", "layer"),
        ],
        "2 blocks + GRU, mean over the beat": two
        + [
            gru_box,
            diagrams.Block(
                "mean over the states\nthat hold the beat", "(B, 64)", "pool"
            ),
            diagrams.Block("Linear\n64 → 3", "(B, 3)", "layer"),
        ],
    }
)

**Setup**: `EPOCHS = 12` epochs and `REPEATS` runs for every arm, as for the reference.
A GRU starts more slowly than a kernel stack; the curves show whether its score on unseen
patients has stopped rising by epoch 12.

**Before you run the next cell**, write down what you expect of each arm against the
control and against `Kernels(5)`, and how many times the seconds of the control you expect
a GRU arm to take.

In [ ]:
reads = ("flatten", "GRU, last state", "GRU, mean over the beat")
state = {
    f"2 blocks + {read}": hb.spread(
        lambda read=read: TwoBlocksThen(read), train, valid, EPOCHS, REPEATS
    )
    for read in reads
}
print(
    f"{hb.line('Kernels(5), the result to beat', reference)}   {hb.n_params(hb.Kernels(5)):6d} weights"
)
for read in reads:
    name = f"2 blocks + {read}"
    versus = None if read == "flatten" else state["2 blocks + flatten"]
    print(
        f"{hb.line(name, state[name], versus)}   {hb.n_params(TwoBlocksThen(read)):6d} weights"
    )
print("pt: the difference with 2 blocks + flatten, the control")
for read in reads[1:]:
    name = f"2 blocks + {read}"
    print(f"{name} against Kernels(5): {hb.points(state[name], reference):+.1f} pt")
hb.curves(
    {"Kernels(5)": reference, **state},
    "Two kernel blocks and three ways on, for EPOCHS epochs",
    floor=nearest["macro_recall"],
)

Compare with what you wrote down, the seconds included.

- **The curves first.** All arms have flattened their validation macro recall by about
  epoch 9, and the train losses of the GRU arms end close to the control's (about 0.045):
  12 epochs gave the GRU the time it needed.
- **The GRU against the control**: the last state ends just under it, the mean over the
  beat just over it, each closer to it than the spread between runs of the same kernels
  with a flatten head. The mean over the beat scored a little higher than the last state
  (about 0.55 against 0.53). Nor would a gap between them say that the tail was the cause:
  the mean also averages about 26 states instead of reading one.
- **Against `Kernels(5)`** both GRU arms are lower, the last state further than the mean
  over the beat.
- **The cost**: a GRU arm trains in about 34 s per run against about 7.5 s for the control
  with the same kernels: four to five times as long, for 2.5 times the weights. A
  plausible cause, not measured here: the 46 steps that wait for each other.

So a state added nothing over a flatten head that already sees every position of the beat.
Whether order matters on a beat at all, this experiment does not settle: every arm here can
use where in the beat something happened. What it suggests about the data: an S beat is an
S because it comes too early after the beat before it. A row holds the beat and, within 1.2 times
the usual time between two beats (RR), the next R-peak: the second spike near the end of the N beats in 2.1's pictures. It
does not hold the beat before. The hypothesis this leaves, not tested here: a design that
sees several beats in a row, in order, gives a state something to carry that a flatten head
cannot reach. `Kernels(5)` finds about 2% of the S beats.

## 2.2.3 The gate: letting parts through

Every block so far treats all of its input alike: a kernel weighs its window with the same
weights at every position, for every beat. A gate lets the network decide, per beat, which
parts of its input pass.

**The sigmoid** `σ(z) = 1 / (1 + e^(−z))` squeezes any number into the range 0 to 1:
large negative numbers go to about 0, large positive ones to about 1, and 0 goes to 0.5.
Lesson 1 met it as an activation and as the per-class probability of `BCEWithLogitsLoss`.
Here it is used as a multiplier.

**A gate** computes such a multiplier for every element of its input, from that input,
and multiplies: `y = σ(W x + b) ⊙ x`, where `⊙` is element by element. At every position a
`Linear(32, 32)` turns the 32 channels into 32 numbers, the sigmoid makes each one a
multiplier between 0 and 1, and each channel is multiplied by its own. Where the gate is
open (about 1) the channel passes; where it is closed (about 0) the output is zero. The
shape stays. Below, the gate sits on the output of three kernel blocks, `(B, 23, 32)`: 23
positions with 32 channels each, and gives `(B, 23, 32)` (slides 12 and 14). Because the multiplier is
computed from the input, it differs per beat and per position.

The same multiplier, put between two routes, is part of other blocks (slides 12 and 16):

| block | formula | the gate Γ chooses, per element, between |
|---|---|---|
| GRU | `h_t = Γ ⊙ h̃_t + (1 − Γ) ⊙ h_{t−1}` | a new candidate state `h̃_t` and the old state |
| highway | `y = Γ ⊙ T(x) + (1 − Γ) ⊙ x` | the block's output and its own input |
| skip | `y = T(x) + x` | nothing: there is no gate, both routes always pass |

The GRU of 2.2.2 is the first row: its gate lets it keep the old state (Γ near 0:
remember) or take the new candidate (Γ near 1: overwrite), element by element. That is how
a GRU holds on to something across many steps. The candidate `h̃_t` is computed like the
simple `f` of 2.2.2, from the new input and the old state, through a tanh. A GRU has a
second gate, the reset gate, which decides how much of the old state goes into that
candidate. PyTorch writes the update gate as `z = 1 − Γ`: the same gate, counted from the
other side.

A **highway** is a skip with a gate on it, from the same year as ResNet (Srivastava and
colleagues, 2015):

<svg width="600" height="190" viewBox="0 0 600 190" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="600" height="190" fill="white"/>
<defs><marker id="a3" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="84" fill="#222">x</text>
<line x1="25" y1="80" x2="178" y2="80" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<rect x="180" y="58" width="120" height="44" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="240" y="85" text-anchor="middle" fill="#222">block T</text>
<line x1="300" y1="80" x2="382" y2="80" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<rect x="384" y="64" width="48" height="32" rx="8" fill="#f3d9e6" stroke="#555"/><text x="408" y="85" text-anchor="middle" fill="#222">× Γ</text>
<path d="M 100 80 L 100 150 L 382 150" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<rect x="384" y="134" width="64" height="32" rx="8" fill="#f3d9e6" stroke="#555"/><text x="416" y="155" text-anchor="middle" fill="#222">× (1−Γ)</text>
<line x1="432" y1="80" x2="485" y2="106" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<line x1="448" y1="150" x2="485" y2="124" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<circle cx="500" cy="115" r="16" fill="#dccdf0" stroke="#555"/><text x="500" y="121" text-anchor="middle" font-size="18" fill="#222">+</text>
<line x1="516" y1="115" x2="568" y2="115" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<text x="574" y="119" fill="#222">y</text>
<path d="M 60 80 L 60 26 L 92 26" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#a3)"/>
<text x="96" y="30" fill="#222">Γ = σ(W x + b): one multiplier per element, from x</text>
<line x1="408" y1="36" x2="408" y2="62" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#a3)"/>
<text x="250" y="170" text-anchor="middle" fill="#222">x passes as it is</text>
</svg>

Where Γ is 1 the block's output passes, where it is 0 the input passes unchanged; the skip
is the same picture with both routes always fully open. vectormesh has it as `Highway`;
lesson 3 builds on this line of blocks.

### 2.2.3.1 What does a trained gate let through?

**Question**: does a gate on the output of a kernel stack pay, and what does it learn to
let through?

**Arms**: the first three blocks of the kernel stack, `hb.stack(3)`, with a flatten head,
with and without a `Gate` between the stack and the head; the rest is the same, so the one
without is the gate's ungated twin. Three blocks, not five: after three blocks 23 positions
are left, each seeing 36 raw steps, enough to see *where* in the beat a gate opens; after
five only 5 positions are left.

**Before you run the next cell**, write down whether you expect the gate to pay, and where
in an N beat and a V beat you expect it to be open.

In [ ]:
class ThreeBlocks(nn.Module):
    # hb.stack(3), (B, 187, 1) -> (B, 23, 32), a gate or nothing, a flatten head

    def __init__(self, gated: bool, width: int = 32):
        super().__init__()
        self.features = nn.Sequential(*hb.stack(3, width))
        self.gate = (
            Gate(width) if gated else nn.Identity()
        )  # (B, 23, 32) -> (B, 23, 32)
        self.head = nn.Linear(23 * width, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.gate(self.features(x)).flatten(1))


three = kernel_blocks(3)
flat = [
    diagrams.Block("flatten", "(B, 736)", "pool"),
    diagrams.Block("Linear\n736 → 3", "(B, 3)", "layer"),
]
draw(
    {
        "3 blocks, no gate": three + flat,
        "3 blocks + gate": three
        + [diagrams.Block("Gate\nσ(W x + b) ⊙ x", "(B, 23, 32)", "gate")]
        + flat,
    }
)

gated_models = []  # the gated models the spread trains, kept to read their gates


def make_gated() -> ThreeBlocks:
    gated_models.append(ThreeBlocks(gated=True))
    return gated_models[-1]


ungated = hb.spread(lambda: ThreeBlocks(gated=False), train, valid, EPOCHS, REPEATS)
gated = hb.spread(make_gated, train, valid, EPOCHS, REPEATS)
print(
    f"{hb.line('3 blocks, no gate', ungated)}   {hb.n_params(ThreeBlocks(False)):6d} weights"
)
print(
    f"{hb.line('3 blocks + gate', gated, ungated)}   {hb.n_params(ThreeBlocks(True)):6d} weights"
)
print(f"3 blocks + gate against Kernels(5): {hb.points(gated, reference):+.1f} pt")
hb.curves(
    {"3 blocks, no gate": ungated, "3 blocks + gate": gated},
    "A gate and its ungated twin, for EPOCHS epochs",
)

The gate costs 1056 weights (13,763 against 12,707) and no measurable seconds. It does not
pay: it ends a little under its ungated twin, closer to it than the spread between their
runs. The two curves lie on top of each other, both flat from about epoch 8, and both stay
below `Kernels(5)`, as depth 3 did in 2.1.5.
The score does not show what the gate does; its multipliers do.

### 2.2.3.2 Reading the gate

The gate's multipliers can be read off a trained model: `σ(W x + b)` for the features `x`
of a beat, one number between 0 and 1 per channel and position. The figure below shows
them for one N beat and one V beat of validation, for the first of the trained gated models.
A multiplier only matters where its channel carries something: where the input to the
gate is near zero, open or closed lets nothing through either way. Those cells are left
white (the 25% of cells with the smallest input). The bottom panel is the mean gate over
all validation V beats minus the mean over all validation N beats: red where the gate is more open for
a V, blue where it is more open for an N. Even a gate with random weights opens differently
on different beats, since its multipliers are computed from them, so part of any such
picture is the features', not the gate's. The last lines of output compare the trained
gate's V − N with that of five gates with random weights on the same trained features.

In [ ]:
model = gated_models[0].cpu().eval()


@torch.no_grad()
def openness(beats: torch.Tensor, model: ThreeBlocks = model):
    # the features the gate sees and its multipliers, both (B, 23, 32)
    features = model.features(beats)
    return features, torch.sigmoid(model.gate.project(features))


def holds_beat(beats: torch.Tensor) -> torch.Tensor:
    # (B, 23): True where the 8 raw steps under a position are not all padding
    return beats[:, : 23 * 8, 0].reshape(len(beats), 23, 8).abs().sum(-1) > 0


labels = valid.labels.numpy()
_, every = openness(valid.beats)  # (n, 23, 32)
inside = holds_beat(valid.beats)  # (n, 23)
order = every[inside].mean(dim=0).argsort()  # channels from most closed to most open
edges = (
    np.arange(24) * 8 / 125
)  # position edges in seconds: each position is 8 raw steps

fig, ax = plt.subplot_mosaic(
    [["N", "V"], ["gate N", "gate V"], ["V - N", "V - N"]],
    figsize=(13, 10),
    height_ratios=[1, 2, 2],
    layout="constrained",
)
for k in (0, 2):
    name = hb.CLASSES[k]
    i = np.flatnonzero(labels == k)[0]
    beat = valid.beats[i : i + 1]
    features, gate = openness(beat)
    shown = gate[0][:, order].T.numpy().copy()  # (32 channels, 23 positions)
    size = features[0][:, order].T.abs().numpy()
    shown[:, ~holds_beat(beat)[0].numpy()] = np.nan  # past the end of the beat
    shown[size < np.quantile(size, 0.25)] = np.nan  # input to the gate near zero
    ax[name].plot(np.arange(hb.STEPS) / 125, beat[0, :, 0], color="0.3")
    ax[name].set_xlim(0, edges[-1])
    ax[name].set_title(f"one {name} beat of validation")
    image = ax[f"gate {name}"].pcolormesh(
        edges, np.arange(33), shown, cmap="viridis", vmin=0, vmax=1
    )
    ax[f"gate {name}"].set_title(
        f"the gate on that {name} beat (white: no beat, or input near zero)"
    )
    ax[f"gate {name}"].set_xlabel("seconds after the R-peak")
    ax[f"gate {name}"].set_ylabel("channel (sorted by mean openness)")
fig.colorbar(image, ax=[ax["gate N"], ax["gate V"]], label="gate: 0 closed, 1 open")


def v_minus_n(gates: torch.Tensor) -> torch.Tensor:
    # the mean gate over the validation V beats minus over the N beats, at each position
    # only over the beats that still hold a beat there: (23, 32)
    means = []
    for k in (2, 0):
        weight = inside[labels == k].float().unsqueeze(-1)
        means.append(
            (gates[labels == k] * weight).sum(dim=0) / weight.sum(dim=0).clamp(min=1)
        )
    return means[0] - means[1]


difference = v_minus_n(every)[:, order].T.numpy()
gap = np.abs(difference).max()
diff_image = ax["V - N"].pcolormesh(
    edges, np.arange(33), difference, cmap="RdBu_r", vmin=-gap, vmax=gap
)
ax["V - N"].set_title(
    "mean gate over the validation V beats minus the mean over the validation N beats, where they hold a beat: red is more open for V"
)
ax["V - N"].set_xlabel(
    "seconds after the R-peak (each column is one of the 23 positions)"
)
ax["V - N"].set_ylabel("channel (sorted)")
fig.colorbar(diff_image, ax=ax["V - N"], label="V − N")
fig.suptitle("Where a trained gate is open")
plt.show()
held = every[inside]  # every gate value at a position that holds a beat
print(
    f"inside the validation beats: {(held > 0.9).float().mean():.2f} of the gate values above 0.9, {(held < 0.1).float().mean():.2f} below 0.1"
)
with torch.no_grad():
    trained_features = model.features(valid.beats)  # the features the trained gate sees
    random_gates = []
    for seed in range(5):
        torch.manual_seed(seed)
        random_gate = Gate(32)  # the same gate, random weights, on the same features
        random_gates.append(torch.sigmoid(random_gate.project(trained_features)))
print("           mean |V − N|   largest |V − N|")
print(
    f"trained    {v_minus_n(every).abs().mean():.3f}          {v_minus_n(every).abs().max():.3f}"
)
for seed, gates in enumerate(random_gates):
    print(
        f"random {seed}   {v_minus_n(gates).abs().mean():.3f}          {v_minus_n(gates).abs().max():.3f}"
    )

**Is the gate open everywhere?** No. Inside the beats of validation about a third of the
gate values are above 0.9, 6% are below 0.1, and the rest lie in between. In the two middle
panels several channels are closed (dark) over the flat stretch between about 0.3 and
0.8 s and open (yellow) in the first 0.2 s, around the R-peak; other channels stay open
over the whole beat. The gate chooses per position: it passes the start of the beat and
holds part of the flat middle back.

**Does it open differently for a V than for an N?** For some channels, yes. In the first
0.4 s, where a V beat is wide and an N beat has a narrow spike, a few channels are up to
0.3 more open for V and others up to 0.3 more closed; from about 0.5 s on, a band of
channels is more closed for V. The columns after about 1.2 s average over the few long
beats that still hold a beat there, and are noisy. A random gate on the same trained
features differs between V and N almost as much: about 0.06 on average over the cells in
all five seeds (largest 0.23 to 0.32), against 0.08 (largest 0.31) for the trained gate.
So most of the structure in the bottom panel is in the features the gate sees: they differ
between V and N, and any gate computed from them opens differently too. Training added a
little on top, about 0.02 on average.

So the trained gate passes different parts of different beats, much of that difference
comes with its features, and the macro recall did not move. The flatten head of the ungated twin has its own weight for every channel at
every position, so it can learn to ignore the flat middle as well, with fixed weights. A
gate can only add something where what to pass has to change with the input in a way fixed
weights cannot follow; on this score, here, that did not show.

## 2.2.4 A gate on whole channels: squeeze-excite

The gate of 2.2.3 decides per channel *and* per position. Squeeze-excite decides per
channel only, for the whole beat, from a summary of all channels (slide 15).

Why that could matter on a heartbeat: a channel of a kernel stack is one learned
detector, a kernel that answers to one shape, say a steep rise, a wide wave or a flat
stretch. Which detectors answer *together* says something about the class: a V beat is
wide and lacks the sharp spike of an N. Squeeze-excite can learn such combinations: when
"wide wave" answers strongly, turn "sharp rise" down and a third detector up. The flatten
head of `Kernels(5)` already sees every channel at every position, but it can only add
them up with fixed weights; it cannot turn one channel down because another one answered.

In 2.1's letters a kernel stack's output is `(B, S, D)`: `S` positions, `D` channels.

- **Squeeze**: 2.1.3's global pool, the mean over the sequence axis, `(B, S, D) -> (B, D)`: every channel
  becomes one number, how strongly it answered anywhere in the beat. *Where* is gone.
- **Excite**: a narrow network, `D -> D/r -> D`, then a sigmoid: one multiplier between 0
  and 1 per channel. `r` is the reduction, here 4, so 32 channels pass through 8 numbers.
  The bottleneck is narrow on purpose: from a summary it can only keep which channels
  usually go together.
- The multipliers scale whole channels: `(B, S, D) * (B, 1, D)`, the same multiplier at
  every position.

`ExcitedKernels` below puts a squeeze-excite in every block of `Kernels(5)`, after the
GELU and before the pool: there each channel is a finished detector answer, and the pool
afterwards keeps what the gate let through.

vectormesh has no squeeze-excite of its own. It is a few lines, and reading them is a test
of whether you can follow the shapes:

In [ ]:
class SqueezeExcite(nn.Module):
    # A gate on whole channels: (B, S, D) -> (B, S, D)

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.excite = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.GELU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        squeezed = x.mean(dim=1)  # (B, S, D) -> (B, D): where is gone
        gate = self.excite(squeezed)  # (B, D), every value between 0 and 1
        return x * gate.unsqueeze(1)  # (B, 1, D) broadcasts over S


se = SqueezeExcite(32)
x = torch.randn(8, 46, 32)
print(f"in {tuple(x.shape)} -> out {tuple(se(x).shape)}, {hb.n_params(se)} weights")


class ExcitedKernels(nn.Module):
    # Kernels(5) with squeeze-excite after the GELU of every block, before its pool

    def __init__(self, width: int = 32):
        super().__init__()
        layers = []
        for layer in hb.stack(5, width):  # kernel, GELU, pool, five times
            layers.append(layer)
            if isinstance(layer, nn.GELU):
                layers.append(SqueezeExcite(width))
        self.features = nn.Sequential(*layers)  # (B, 187, 1) -> (B, 5, 32)
        self.head = nn.Linear(5 * width, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


print(
    f"Kernels(5): {hb.n_params(hb.Kernels(5))} weights, ExcitedKernels: {hb.n_params(ExcitedKernels())}"
)

**Before you run the next cell**, write down what you expect squeeze-excite to add to the
stack of depth 5, and why.

In [ ]:
excited = hb.spread(ExcitedKernels, train, valid, EPOCHS, REPEATS)
print(hb.line("Kernels(5), the result to beat", reference))
print(hb.line("Kernels(5) + squeeze-excite", excited, reference))
print(
    "recall per class, Kernels(5):    ",
    {c: round(r, 2) for c, r in reference.recall().items()},
)
print(
    "recall per class, + squeeze-excite:",
    {c: round(r, 2) for c, r in excited.recall().items()},
)
hb.curves(
    {"Kernels(5)": reference, "Kernels(5) + squeeze-excite": excited},
    "Squeeze-excite on the kernel stack, for EPOCHS epochs",
)

Squeeze-excite adds 2760 weights (24,043 against 21,283) and a third to a half to the seconds. Its
macro recall ends under that of `Kernels(5)`, and the curves show that it does not help: it starts more slowly and
does not rise above the reference in any of the 12 epochs. Per class, V drops a few
hundredths and S stays near zero. A possible reason, not tested here: on 187 steps of one
beat the flatten head already sees which channels answered where, and a per-channel
summary adds little to that.

Lesson 3 takes the line from gate to highway to mixture of experts further (notebook
3.1), and asks the same question of each: what did the extra weights buy?

## 2.2.5 Why S stays at zero: what the blocks cannot see

In every arm of this notebook the recall of S stays near 0.02, whatever block went in: a
skip, a state, a gate, squeeze-excite. No block changes that, so the cause is not in the
blocks. It is in what the row contains, and to see that you need to know what an S beat
is.

**The heart, in three sentences.** A normal beat (N) starts in the sinus node in the
atria, travels through the ventricles by the usual route, and gives the narrow spike of the
R-peak. A **ventricular** ectopic beat (V) starts inside a ventricle: the signal takes an
unusual route, and the spike is wide and has another shape, which a kernel sees. A
**supraventricular** ectopic beat (S) starts too early somewhere above the ventricles, in
the atria or at the junction, and then travels the usual route: the spike looks almost like
an N. What gives an S away is **when** it comes (too early after the beat before it) and
the wave that precedes it (the P wave of the atria, which is different or missing).

That fits the recall per class: V about 0.7, S about 0.02. And it says what a row lacks.
A row starts at the R-peak, so the P wave is not in it, and neither is the beat before it,
so the time since that beat cannot be seen. A kernel, a state and a gate work on the row
they get.

**The hypothesis**, which the data can test: the interval before a beat, in comparison with
this patient's usual interval, separates S from N. The companion dataset
`pttrn-io/ecg-mitbih-10s-raw` holds the same recordings uncut: windows of 10 seconds, every
beat with its R-peak and class. The next cell computes for every beat the interval to the
beat before it, divided by the median interval of that patient (the patient's own normal
rhythm, from the R-peaks only, no labels), and draws it per class.

In [ ]:
from vectormesh import VectorCache

windows = VectorCache.from_hub("pttrn-io/ecg-mitbih-10s-raw", split="train")
rows = windows.dataset.with_format(None)
w_record, w_start = np.asarray(rows["record"]), np.asarray(rows["start"])


def interval_before() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # per beat of train: its record, its class, and the interval to the beat before it
    # divided by the median interval of that record; (n,), (n,), (n,)
    records, classes, ratio = [], [], []
    for record in np.unique(w_record):
        rows_of = np.flatnonzero(w_record == record)
        peaks = np.concatenate(
            [w_start[i] + np.asarray(rows["r_peaks"][i], dtype=int) for i in rows_of]
        )
        kinds = np.concatenate(
            [np.asarray(rows["beat_class"][i], dtype=int) for i in rows_of]
        )
        gaps = np.diff(peaks)
        records.append(np.full(len(peaks), record))
        classes.append(kinds)
        ratio.append(np.r_[np.nan, gaps] / np.median(gaps))  # the first beat has none
    return np.concatenate(records), np.concatenate(classes), np.concatenate(ratio)


beat_record, beat_class, early = interval_before()
fig, ax = plt.subplots(figsize=(9, 3.6))
for k, name in enumerate(hb.CLASSES):
    ax.hist(
        early[beat_class == k],
        bins=np.linspace(0.2, 1.8, 65),
        density=True,
        histtype="step",
        lw=2,
        label=f"{name}  (median {np.nanmedian(early[beat_class == k]):.2f})",
    )
ax.axvline(1, color="0.5", ls="--", lw=1)
ax.set_xlabel("interval before the beat / this patient's median interval")
ax.set_ylabel("density")
ax.set_title("Does a beat come too early? 1 = on time, below 1 = early")
ax.legend()
plt.tight_layout()
plt.show()
for k, name in enumerate(hb.CLASSES):
    share = np.nanmean(early[beat_class == k] < 0.85)
    print(f"{name}: {share:.0%} of the beats come more than 15% early")

**Before you read on**, write down what you expect: where does the histogram of S lie, and
how much of it overlaps with N?

In [ ]:
# One rule, no model: a beat that comes more than 15% early is called S or V. Scored on
# the six validation patients, with the intervals computed from all their beats.
on_valid = np.isin(beat_record, hb.VAL_RECORDS)
flagged = early[on_valid] < 0.85
for k, name in enumerate(hb.CLASSES):
    found = np.nanmean(flagged[beat_class[on_valid] == k])
    print(
        f"{name}: the rule flags {found:.0%} of the {int((beat_class[on_valid] == k).sum())} beats"
    )

Compare with what you wrote down.

- **S comes early**: its median lies at about 0.63 of the patient's own interval, N at 1.0,
  and 91% of the S beats come more than 15% early against 5% of the N beats. V lies in
  between (median 0.72, 70% early), and V has the shape a kernel already sees.
- **One rule, no model** (the second cell): on the validation patients a beat that comes
  more than 15% early flags about 88% of the S beats. The kernel stack finds 2%. The rule
  pays for that with false alarms: it flags about 8% of the N beats, about twice as
  many beats as there are S, and it flags V as well. It is not a classifier; it is
  evidence that the information is in the interval before a beat and not in the beat.

So the blocks of this notebook did not fail on S: none of them was given what separates S
from N. Which block you put on a row matters less than what is in the row, and what should
be in the row is **domain knowledge**: what a cardiologist looks at, the rhythm and not
only the shape. A design can get the interval in as an extra input, as a second sequence
(the intervals of the last few beats), or by giving the model more than one beat. How you
do that is the question of the exercise (notebook 2.4); the window dataset above is
the data for it. Two cautions: the median interval here uses the whole recording of a
patient, which a model on a live patient does not have, and the exercise asks what a
calibration with only the first minute would change; and a recall of S bought with a flood
of false alarms on N is a cost that macro recall does not show.

## 2.2.6 What you now know

- **Skip.** A plain stack of 16 blocks does not learn here: every block shrinks the
  differences between beats by about a factor 3, until the head sees the same numbers for
  every beat and the train loss stays at the level of the class-share loss; the gradient
  shrinks the same way on its way back. A skip, `y = T(x) + x`, keeps the shape and adds
  each block's input back, so the differences between beats pass every block: with
  `PlainSkip` and with vectormesh's `Skip` the 16-block stack trains from the first epoch.
  A LayerNorm before every block, without the add, fixed the start but collapsed in
  training; the add, not the norm inside `Skip`, is what made the stack train. Depth itself
  bought nothing on these beats: 16 skip blocks are level with 2 plain blocks, at six times
  the weights.
- **State.** A recurrent layer walks the axis and keeps a state of `H` numbers, updated
  from the new input and the old state with the same weights at every step, in its simplest
  form `h_t = tanh(W x_t + U h_{t-1} + b)`: `(B, 46, 32) -> (B, 46, 64)`,
  `[:, -1] -> (B, 64)`, `Linear -> (B, 3)`. It can relate parts at any distance, in order;
  it costs steps that wait for each other (four to five times the seconds here), and the
  past fades.
- **The state on one beat** added nothing over a flatten head that sees every position,
  read from the last state or from the mean over the beat. Whether order matters on a beat
  is not settled; a row lacks the beat before, and a design over several beats is the
  hypothesis left.
- **Gate.** A gate multiplies every element by a number between 0 and 1 computed from the
  input; the trained gate here closes channels over the flat middle of the beat and opens
  differently for V than for N, but a random gate on the same features opens differently
  almost as much (mean |V − N| about 0.06 against 0.08): most of that difference is the
  features'. The gate did not move the macro recall. GRU and highway put such a gate between two routes; the skip is the two routes
  without a gate.
- **Squeeze-excite** gates whole channels, from a summary of all channels (a global pool) through a
  bottleneck; on the depth-5 stack it did not help, and may cost a little.
- The result to beat is still `Kernels(5)`.
- **S stays at zero in every arm**, because none of the blocks sees what makes an S: it
  comes too early after the beat before, and a row does not hold that beat. On the
  uncut recordings 91% of the S beats come more than 15% early against 5% of the N beats,
  and a rule on that one number flags 88% of the S beats on validation, with false alarms
  on N. What goes into the row is a matter of domain knowledge, and the exercise (2.4) is
  where you design for it.